In [ ]:
!pip install textstat nltk pandas tqdm matplotlib seaborn wordcloud

### Import Libraries

In [ ]:
import os
import pandas as pd
import nltk
import textstat
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter

# Download the Punkt tokeniser models required for sentence splitting later
nltk.download('punkt')
nltk.download('punkt_tab')

print("Libraries loaded successfully.")

### Load Raw Documents
Read every .txt file from the dataset folder into a single pandas DataFrame. Each row represents one document. Company name and document type are extracted directly from the filename convention CompanyName_DocumentType.txt.

In [ ]:
# Full path to the folder containing all raw .txt policy documents
DATA_PATH = r"C:\Users\HP\Downloads\archive\text"

records = []

for filename in tqdm(os.listdir(DATA_PATH), desc="Loading files"):

    # Skip any non-.txt files that may exist in the folder
    if not filename.endswith('.txt'):
        continue

    filepath = os.path.join(DATA_PATH, filename)

    # Attempt UTF-8 decoding first — fall back to latin-1 for files
    # with non-standard encoding, which is common in scraped web data
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    except UnicodeDecodeError:
        with open(filepath, 'r', encoding='latin-1') as f:
            text = f.read()

    # Exclude documents with negligible content — not useful for analysis
    if len(text.strip()) < 50:
        continue

    # Parse filename to extract company name and document type
    # e.g. "1Password_PrivacyPolicy.txt" -> company="1Password", doc_type="PrivacyPolicy"
    name_without_ext = filename.replace('.txt', '')
    parts    = name_without_ext.split('_', 1)
    company  = parts[0] if len(parts) >= 1 else name_without_ext
    doc_type = parts[1] if len(parts) == 2 else 'Unknown'

    records.append({
        'filename'   : filename,
        'company'    : company,
        'doc_type'   : doc_type,
        'text'       : text,
        'char_count' : len(text),
        'word_count' : len(text.split()),
    })

df = pd.DataFrame(records)

print(f"Total files found in folder  : {len(os.listdir(DATA_PATH))}")
print(f"Total valid documents loaded : {len(df)}")

### Basic Datframe Overview
Inspect the shape, column names, and a sample of rows to confirm the data loaded correctly.

In [ ]:
print(f"Shape of the dataframe: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()
print("First 5 rows:")
df[['filename', 'company', 'doc_type', 'word_count']].head()

### Document Type Distribution
Examine how documents are distributed across policy types. This reveals whether the dataset is dominated by one category and informs the document type comparison we will conduct during analysis.

In [ ]:
doc_type_counts = df['doc_type'].value_counts()

print("Top 20 document types:")
print(doc_type_counts.head(20))

plt.figure(figsize=(12, 5))
doc_type_counts.head(15).plot(kind='bar', color='steelblue', edgecolor='white')
plt.title("Top 15 Document Types")
plt.xlabel("Document Type")
plt.ylabel("Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Document Length Distribution
Examine the distribution of document lengths by word count. A right-skewed distribution is expected in legal text corpora — a small number of extremely long documents is normal. The top 2% is clipped for visual clarity.

In [ ]:
print("Word count summary:")
print(df['word_count'].describe())

plt.figure(figsize=(12, 4))
df['word_count'].clip(upper=df['word_count'].quantile(0.98)).hist(
    bins=60, color='steelblue', edgecolor='white'
)
plt.title("Document Length Distribution (word count, top 2% clipped)")
plt.xlabel("Word Count")
plt.ylabel("Number of Documents")
plt.tight_layout()
plt.show()

### Companies with the Most Documents
Identify which companies contributed the most documents to the dataset. Companies with many documents may have an outsized influence on the analysis.

In [ ]:
company_counts = df['company'].value_counts()

print("Companies with most documents:")
print(company_counts.head(15))

plt.figure(figsize=(12, 4))
company_counts.head(20).plot(kind='bar', color='steelblue', edgecolor='white')
plt.title("Top 20 Companies by Number of Documents")
plt.xlabel("Company")
plt.ylabel("Number of Documents")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Flesch Reading Ease Score Distribution
Compute the Flesch Reading Ease score for every document. This score is the basis for our readability labels. Visualising the distribution before labelling is essential — it allows us to set thresholds that reflect the actual spread of scores in this specific corpus rather than assuming a universal scale.

In [ ]:
print("Computing Flesch Reading Ease scores — this may take a few minutes...")

df['flesch_score'] = df['text'].apply(lambda x: textstat.flesch_reading_ease(x))

print("Flesch Score Summary:")
print(df['flesch_score'].describe())

plt.figure(figsize=(12, 4))
df['flesch_score'].clip(-50, 100).hist(bins=60, color='steelblue', edgecolor='white')
plt.axvline(x=30, color='red',   linestyle='--', label='Threshold: below 30 = Confusing')
plt.axvline(x=60, color='green', linestyle='--', label='Threshold: above 60 = Clear')
plt.title("Flesch Reading Ease Score Distribution")
plt.xlabel("Flesch Score")
plt.ylabel("Number of Documents")
plt.legend()
plt.tight_layout()
plt.show()

print()
print(f"Below 30 (Confusing) : {len(df[df['flesch_score'] < 30])}")
print(f"30 to 60 (Middle)    : {len(df[(df['flesch_score'] >= 30) & (df['flesch_score'] <= 60)])}")
print(f"Above 60 (Clear)     : {len(df[df['flesch_score'] > 60])}")

### Filter Out Documents with Broken Flesch Scores
Some documents contain malformed text with no punctuation or extremely long unbroken strings. These produce nonsensical Flesch scores (e.g. -96,590) that would corrupt the analysis. We remove any document scoring below -100 as these are clearly parsing failures rather than genuine readability measurements.

In [ ]:
# Record how many documents exist before filtering
docs_before = len(df)

# Remove documents with clearly broken Flesch scores caused by malformed text
df = df[df['flesch_score'] >= -100].copy()

print(f"Documents before filtering : {docs_before}")
print(f"Documents after filtering  : {len(df)}")
print(f"Documents removed          : {docs_before - len(df)}")

### Normalise Document Types
The raw dataset contains many variations of the same document type due to inconsistent capitalisation during scraping — for example PrivacyPolicy, Privacypolicy, PRIVACYPOLICY, and Privacy all refer to the same category. We normalise these into a consistent set of broad categories to enable meaningful document type comparisons during analysis.

In [ ]:
# Convert all doc_type values to lowercase to remove capitalisation inconsistencies
df['doc_type_clean'] = df['doc_type'].str.lower().str.strip()

# Map all variations into a standard set of broad categories
# Any type not matching a known pattern is labelled as 'other'
def normalise_doc_type(doc_type):
    if 'privacy' in doc_type:
        return 'privacy_policy'
    elif 'terms' in doc_type or 'tos' in doc_type or 'conditions' in doc_type:
        return 'terms_of_service'
    elif 'cookie' in doc_type:
        return 'cookie_policy'
    elif 'acceptable' in doc_type or 'aup' in doc_type:
        return 'acceptable_use_policy'
    else:
        return 'other'

df['doc_type_clean'] = df['doc_type_clean'].apply(normalise_doc_type)

print("Normalised document type counts:")
print(df['doc_type_clean'].value_counts())

### Apply Percentile-Based Readability Labels
Rather than using fixed thresholds (below 30 = Confusing, above 60 = Clear), we use the actual distribution of this corpus to determine label boundaries. Documents in the bottom 25% of Flesch scores are labelled Confusing and documents in the top 25% are labelled Clear. The middle 50% is dropped. This approach guarantees balanced classes and is more appropriate than fixed thresholds given that the majority of scores in this corpus fall between 20 and 50.

In [ ]:
# Compute the 25th and 75th percentile cutoffs from the actual score distribution
lower_threshold = df['flesch_score'].quantile(0.25)
upper_threshold = df['flesch_score'].quantile(0.75)

print(f"Lower threshold (25th percentile) : {lower_threshold:.2f}")
print(f"Upper threshold (75th percentile) : {upper_threshold:.2f}")

# Assign labels based on percentile thresholds
df_confusing = df[df['flesch_score'] <= lower_threshold].copy()
df_confusing['label']      = 1
df_confusing['label_name'] = 'Confusing'

df_clear = df[df['flesch_score'] >= upper_threshold].copy()
df_clear['label']      = 0
df_clear['label_name'] = 'Clear'

# Combine both labelled groups and drop the middle band
df_labelled = pd.concat([df_confusing, df_clear], ignore_index=True)

confusing_pct = len(df_labelled[df_labelled['label'] == 1]) / len(df_labelled) * 100
clear_pct     = len(df_labelled[df_labelled['label'] == 0]) / len(df_labelled) * 100

print(f"Total labelled documents : {len(df_labelled)}")
print(f"Confusing (label 1)      : {len(df_labelled[df_labelled['label'] == 1])} ({confusing_pct:.1f}%)")
print(f"Clear (label 0)          : {len(df_labelled[df_labelled['label'] == 0])} ({clear_pct:.1f}%)")

### Visualise Final Label Distribution
Confirm the class balance visually before proceeding to sentence splitting.

In [ ]:
plt.figure(figsize=(6, 4))
df_labelled['label_name'].value_counts().plot(
    kind='bar', color=['steelblue', 'coral'], edgecolor='white'
)
plt.title("Label Distribution After Percentile-Based Thresholding")
plt.xlabel("Label")
plt.ylabel("Number of Documents")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Split Documents Into Sentences
Each document is split into individual sentences using the NLTK Punkt tokeniser. Every sentence inherits its parent document's label, company name, document type, and Flesch score. Sentences shorter than 5 words are discarded as they carry no meaningful linguistic signal.


In [ ]:
sentence_records = []

for _, row in tqdm(df_labelled.iterrows(), total=len(df_labelled), desc="Splitting into sentences"):

    # Tokenise document text into a list of individual sentences
    sentences = nltk.sent_tokenize(row['text'])

    for sentence in sentences:
        sentence = sentence.strip()

        # Discard sentences that are too short to carry meaningful content
        if len(sentence.split()) < 5:
            continue

        sentence_records.append({
            'filename'      : row['filename'],
            'company'       : row['company'],
            'doc_type'      : row['doc_type_clean'],
            'sentence'      : sentence,
            'label'         : row['label'],
            'label_name'    : row['label_name'],
            'flesch_score'  : row['flesch_score']
        })

df_sentences = pd.DataFrame(sentence_records)

print(f"Total sentences created  : {len(df_sentences)}")
print(f"Confusing sentences      : {len(df_sentences[df_sentences['label'] == 1])}")
print(f"Clear sentences          : {len(df_sentences[df_sentences['label'] == 0])}")
df_sentences[['company', 'doc_type', 'sentence', 'label_name']].head(5)

### Clean Sentence Text
The raw sentences contain newline characters (\n) and other whitespace artifacts inherited from the scraped web text. These must be removed before sentiment scoring and BERT tokenisation as they introduce noise that affects both analyses. We also strip any remaining leading or trailing whitespace from each sentence.

In [ ]:
# Replace all newline and carriage return characters with a single space
# This handles both Unix-style \n and Windows-style \r\n line endings
df_sentences['sentence'] = df_sentences['sentence'].str.replace(r'[\n\r]+', ' ', regex=True)

# Collapse multiple consecutive spaces into one — a side effect of newline removal
df_sentences['sentence'] = df_sentences['sentence'].str.replace(r' +', ' ', regex=True)

# Strip leading and trailing whitespace from every sentence
df_sentences['sentence'] = df_sentences['sentence'].str.strip()

# Drop any sentences that became too short after cleaning
# (some sentences were mostly whitespace or newlines)
df_sentences = df_sentences[df_sentences['sentence'].apply(lambda x: len(x.split()) >= 5)].reset_index(drop=True)

print(f"Total sentences after cleaning  : {len(df_sentences)}")
print(f"Confusing sentences             : {len(df_sentences[df_sentences['label'] == 1])}")
print(f"Clear sentences                 : {len(df_sentences[df_sentences['label'] == 0])}")

print()
print("Sample sentences after cleaning:")
df_sentences[['company', 'sentence', 'label_name']].head(5)

### Save Both DataFrames to CSV
Save the document-level and sentence-level dataframes to disk. All subsequent analysis notebooks load directly from these files so the raw data never needs to be reprocessed.

In [ ]:
OUTPUT_PATH = r"C:\Users\HP\Downloads\archive"

doc_csv_path      = os.path.join(OUTPUT_PATH, 'documents_labelled.csv')
sentence_csv_path = os.path.join(OUTPUT_PATH, 'sentences_labelled.csv')

# Drop the raw text column from the document-level file to keep the file size manageable
df_labelled.drop(columns=['text']).to_csv(doc_csv_path, index=False)
df_sentences.to_csv(sentence_csv_path, index=False)

print(f"documents_labelled.csv saved  : {len(df_labelled)} rows")
print(f"sentences_labelled.csv saved  : {len(df_sentences)} rows")

Reload both saved CSV files from disk to confirm they saved correctly and the structure looks as expected before moving on to Day 2.

In [ ]:
check_docs      = pd.read_csv(doc_csv_path)
check_sentences = pd.read_csv(sentence_csv_path)

print("Document-level CSV:")
print(f"  Rows    : {len(check_docs)}")
print(f"  Columns : {list(check_docs.columns)}")
print(check_docs[['company', 'doc_type_clean', 'label_name', 'flesch_score']].head(3))

print()
print("Sentence-level CSV:")
print(f"  Rows    : {len(check_sentences)}")
print(f"  Columns : {list(check_sentences.columns)}")
print(check_sentences[['company', 'doc_type', 'sentence', 'label_name']].head(3))

print()
print("Data preparation complete. Ready for Section 2 — Sentiment Analysis.")